# NYC Neighborhood Pulse
## Notebook 01 — Data Loading & Preprocessing

**Goal**: Load Airbnb listings + reviews, merge them, filter & sample, save clean dataset for NLP.

**Output**: `outputs/merged_reviews.csv`

## 0. Install Dependencies

In [ ]:
# Run once to install required packages
# !pip install pandas geopandas langdetect tqdm

## 1. Import & Config

In [ ]:
import pandas as pd
import os
from tqdm import tqdm

# Path Config
BASE_DIR   = os.getcwd()          # folder where this notebook lives
DATA_DIR   = BASE_DIR             # listings.csv & reviews.csv are here
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

LISTINGS_PATH = os.path.join(DATA_DIR, 'listings.csv')
REVIEWS_PATH  = os.path.join(DATA_DIR, 'reviews.csv')

print('Data dir :', DATA_DIR)
print('Output dir:', OUTPUT_DIR)

## 2. Load Listings — keep only needed columns

In [ ]:
LISTING_COLS = [
    'id',
    'neighbourhood_cleansed',       # neighborhood name
    'neighbourhood_group_cleansed', # borough
    'latitude',
    'longitude',
    'room_type',
    'price',
    'number_of_reviews',
]

listings = pd.read_csv(LISTINGS_PATH, usecols=LISTING_COLS, low_memory=False)
listings.rename(columns={
    'id': 'listing_id',
    'neighbourhood_cleansed':       'neighborhood',
    'neighbourhood_group_cleansed': 'borough',
}, inplace=True)

print(f'Listings loaded: {len(listings):,} rows')
listings.head(3)

In [ ]:
# Borough distribution
print('Borough counts:')
print(listings['borough'].value_counts())
print(f'\nUnique neighborhoods: {listings["neighborhood"].nunique()}')

## 3. Load Reviews

In [ ]:
REVIEW_COLS = ['listing_id', 'id', 'date', 'comments']

reviews = pd.read_csv(REVIEWS_PATH, usecols=REVIEW_COLS, low_memory=False)
reviews.rename(columns={'id': 'review_id'}, inplace=True)
reviews['date'] = pd.to_datetime(reviews['date'], errors='coerce')

print(f'Reviews loaded: {len(reviews):,} rows')
reviews.head(3)

## 4. Merge: Reviews ← join → Listings

In [ ]:
df = reviews.merge(
    listings[['listing_id', 'neighborhood', 'borough', 'latitude', 'longitude', 'room_type']],
    on='listing_id',
    how='inner'
)

print(f'Merged dataset: {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 5. Filter — clean text, date range, English only

In [ ]:
print(f'Before filtering: {len(df):,}')

# 5a. Drop nulls & very short comments
df = df.dropna(subset=['comments'])
df = df[df['comments'].str.len() > 50]   # at least 50 chars
print(f'After length filter: {len(df):,}')

# 5b. Keep only recent reviews (2022 onwards) for relevance
df = df[df['date'] >= '2022-01-01']
print(f'After date filter (2022+): {len(df):,}')

In [ ]:
# 5c. Language detection — keep English only
# This step takes ~5 minutes; skip if you want to move fast (most reviews are English)

RUN_LANG_DETECT = False   # ← set True to enable

if RUN_LANG_DETECT:
    from langdetect import detect, LangDetectException

    def safe_detect(text):
        try:
            return detect(str(text))
        except LangDetectException:
            return 'unknown'

    tqdm.pandas(desc='Detecting language')
    df['lang'] = df['comments'].progress_apply(safe_detect)
    df = df[df['lang'] == 'en']
    print(f'After English filter: {len(df):,}')
else:
    print('Language detection skipped — assuming mostly English')

## 6. Stratified Sample — cap at 150k reviews (BERTopic sweet spot)

In [ ]:
SAMPLE_SIZE = 150_000

if len(df) > SAMPLE_SIZE:
    # Stratified by neighborhood so smaller neighborhoods aren't lost
    df = (
        df.groupby('neighborhood', group_keys=False)
          .apply(lambda x: x.sample(
              n=max(1, int(len(x) / len(df) * SAMPLE_SIZE)),
              random_state=42
          ))
    )
    print(f'After stratified sampling: {len(df):,}')
else:
    print(f'No sampling needed, dataset size: {len(df):,}')

df = df.reset_index(drop=True)

## 7. Quick EDA

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Reviews per borough
df['borough'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Reviews per Borough')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

# Top 20 neighborhoods
df['neighborhood'].value_counts().head(20).plot(
    kind='barh', ax=axes[1], color='mediumpurple'
)
axes[1].set_title('Top 20 Neighborhoods by Review Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_review_distribution.png'), dpi=150)
plt.show()
print('Chart saved to outputs/')

In [ ]:
# Neighborhood coverage summary
neigh_stats = (
    df.groupby(['borough', 'neighborhood'])
      .agg(review_count=('review_id', 'count'))
      .reset_index()
      .sort_values('review_count', ascending=False)
)
print(f'Neighborhoods with data: {neigh_stats["neighborhood"].nunique()} / 224')
print(f'Min reviews per neighborhood: {neigh_stats["review_count"].min()}')
print(f'Max reviews per neighborhood: {neigh_stats["review_count"].max()}')
neigh_stats.head(10)

## 8. Save

In [ ]:
out_path = os.path.join(OUTPUT_DIR, 'merged_reviews.csv')
df.to_csv(out_path, index=False)
print(f'✅ Saved: {out_path}')
print(f'   Rows   : {len(df):,}')
print(f'   Columns: {list(df.columns)}')

---
## ✅ Notebook 01 Complete

Output saved to `outputs/merged_reviews.csv`.

**Next**: Run `02_nlp_processing.ipynb` for BERTopic modeling.